# A3 E1-Augmentation: n=50 → n=100 Statistical Power Boost

**Purpose**: Run 50 additional questions (indices 50–99) on the same protocol as E1/E1b  
**Models**: Sonnet 4, GPT-4o, Haiku 3.5, GPT-4o-mini, GPT-3.5-turbo  
**Domains**: GSM8K, CommonsenseQA, BoolQ  
**Lambda**: 0.0, 0.4, 0.8, 1.0  
**Expected trials**: 5 models × 3 domains × (1 direct + 4 wrong_cue) = 75 conditions × 50 questions = 3,750 trials  
**Expected runtime**: ~3-4 hours on Colab Pro+

Merging with E1/E1b gives n=100 per cell → 95% CI narrows ~30%.


In [ ]:
# @title 1. Setup & Configuration
!pip install anthropic openai datasets tqdm -q

import json, os, re, time, random
from datetime import datetime
from typing import List, Optional, Tuple, Any
from dataclasses import dataclass, asdict
from tqdm import tqdm
import numpy as np

# ━━━ CONFIGURATION ━━━
EXPERIMENT_ID = "A3_E1aug"
GLOBAL_SEED = 43  # Different from E1 seed=42
PROBLEM_OFFSET = 50  # Start from index 50
N_PROBLEMS = 50  # 50 new questions per domain
TRACE_LENGTH = 8  # Same as E1

LAMBDA_LEVELS = [0.0, 0.4, 0.8, 1.0]
TEMPERATURE = 0
MAX_TOKENS_CONSUMER = 1024
MAX_TOKENS_TRACE = 2048
API_DELAY = 0.3
API_MAX_RETRIES = 5

MODELS = {
    "sonnet4":   ("anthropic", "claude-sonnet-4-20250514"),
    "gpt4o":     ("openai",    "gpt-4o"),
    "haiku35":   ("anthropic", "claude-haiku-4-5-20251001"),
    "gpt4omini": ("openai",    "gpt-4o-mini"),
    "gpt35":     ("openai",    "gpt-3.5-turbo"),
}
PRODUCER = ("openai", "gpt-4o")  # Trace generator

DOMAINS = ["gsm8k", "csqa", "boolq"]

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# ━━━ API KEYS (set in Colab secrets or paste here) ━━━
import os
from google.colab import userdata

try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')

try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

assert ANTHROPIC_API_KEY, "Set ANTHROPIC_API_KEY in Colab Secrets"
assert OPENAI_API_KEY, "Set OPENAI_API_KEY in Colab Secrets"

import anthropic, openai
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

# ━━━ GOOGLE DRIVE MOUNT ━━━
from google.colab import drive
drive.mount('/content/drive')

# Save to Drive so data persists across sessions
DRIVE_BASE = "/content/drive/MyDrive/A3"
BASE_DIR = f"{DRIVE_BASE}/E1aug"
for d in ["data", "traces", "results", "checkpoints", "figures"]:
    os.makedirs(f"{BASE_DIR}/{d}", exist_ok=True)

print(f"✓ Output directory: {BASE_DIR} (Google Drive)")

print(f"✓ Experiment: {EXPERIMENT_ID}")
print(f"✓ Problem range: [{PROBLEM_OFFSET}, {PROBLEM_OFFSET + N_PROBLEMS - 1}]")
print(f"✓ Models: {list(MODELS.keys())}")
print(f"✓ Lambda levels: {LAMBDA_LEVELS}")
print(f"✓ Expected trials: {len(MODELS) * len(DOMAINS) * (1 + len(LAMBDA_LEVELS)) * N_PROBLEMS}")


In [ ]:
# @title 2. Load Datasets (indices 50-99)
from datasets import load_dataset

def load_gsm8k(offset, n):
    ds = load_dataset("openai/gsm8k", "main", split="test")
    problems = []
    for i in range(offset, min(offset + n, len(ds))):
        item = ds[i]
        # Extract numerical answer from "#### XXXX" format
        answer_text = item["answer"]
        match = re.search(r'####\s*([\-\d,]+)', answer_text)
        final_answer = match.group(1).replace(",", "") if match else "0"
        problems.append({
            "question": item["question"],
            "correct_answer": final_answer,
            "answer_type": "numerical",
            "domain": "gsm8k",
            "problem_index": i,
        })
    return problems

def load_csqa(offset, n):
    ds = load_dataset("tau/commonsense_qa", split="validation")
    problems = []
    for i in range(offset, min(offset + n, len(ds))):
        item = ds[i]
        choices = {}
        for label, text in zip(item["choices"]["label"], item["choices"]["text"]):
            choices[label] = text
        choice_text = " ".join([f"({l}) {t}" for l, t in choices.items()])
        problems.append({
            "question": f"{item['question']}\n{choice_text}",
            "question_stem": item["question"],
            "choices": choices,
            "correct_answer": item["answerKey"],
            "correct_text": choices.get(item["answerKey"], ""),
            "answer_type": "multiple_choice",
            "domain": "csqa",
            "problem_index": i,
        })
    return problems

def load_boolq(offset, n):
    ds = load_dataset("google/boolq", split="validation")
    problems = []
    for i in range(offset, min(offset + n, len(ds))):
        item = ds[i]
        passage = item["passage"][:500]  # Truncate long passages
        correct = item["answer"]  # True/False
        problems.append({
            "question": f"Passage: {passage}\n\nQuestion: {item['question']}",
            "question_stem": item["question"],
            "passage": passage,
            "correct_answer": correct,
            "answer_type": "boolean",
            "domain": "boolq",
            "problem_index": i,
        })
    return problems

# Load all
data = {}
data["gsm8k"] = load_gsm8k(PROBLEM_OFFSET, N_PROBLEMS)
data["csqa"] = load_csqa(PROBLEM_OFFSET, N_PROBLEMS)
data["boolq"] = load_boolq(PROBLEM_OFFSET, N_PROBLEMS)

for domain, problems in data.items():
    with open(f"{BASE_DIR}/data/{domain}.json", "w") as f:
        json.dump(problems, f, indent=2, default=str)
    print(f"✓ {domain}: {len(problems)} problems (indices {problems[0]['problem_index']}–{problems[-1]['problem_index']})")


In [ ]:
# @title 3. API Call Helpers

def call_api(provider, model, system_prompt, user_prompt, max_tokens=1024, temperature=0):
    for attempt in range(API_MAX_RETRIES):
        try:
            if provider == "anthropic":
                resp = claude_client.messages.create(
                    model=model, max_tokens=max_tokens, temperature=temperature,
                    system=system_prompt,
                    messages=[{"role": "user", "content": user_prompt}]
                )
                time.sleep(API_DELAY)
                return resp.content[0].text
            else:  # openai
                resp = openai_client.chat.completions.create(
                    model=model, max_tokens=max_tokens, temperature=temperature,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ]
                )
                time.sleep(API_DELAY)
                return resp.choices[0].message.content
        except Exception as e:
            wait = API_DELAY * (2 ** attempt)
            print(f"  ⚠ {provider}/{model} attempt {attempt+1}: {e}. Retry in {wait:.0f}s")
            time.sleep(wait)
    return "ERROR"

def parse_answer(response, answer_type):
    if not response or response == "ERROR":
        return None
    try:
        # Try JSON parse first
        match = re.search(r'\{[^}]*"final"\s*:\s*"?([^"\}]+)"?', response)
        if match:
            raw = match.group(1).strip()
            if answer_type == "numerical":
                return raw.replace(",", "")
            elif answer_type == "boolean":
                return raw.lower() in ("true", "yes", "1")
            else:
                return raw.upper().strip()

        # Fallback parsing
        if answer_type == "numerical":
            nums = re.findall(r'-?[\d,]+', response)
            return nums[-1].replace(",", "") if nums else None
        elif answer_type == "boolean":
            low = response.lower()
            if "true" in low or "yes" in low:
                return True
            elif "false" in low or "no" in low:
                return False
            return None
        else:  # multiple choice
            match = re.search(r'\b([A-E])\b', response)
            return match.group(1) if match else None
    except:
        return None

print("✓ API helpers ready")


In [ ]:
# @title 4. Generate Reasoning Traces (Producer: GPT-4o)
# This generates clean + corrupt traces for each problem.
# Runtime: ~30-45 min for 150 problems

def make_trace_prompt(problem, domain, target_steps, corrupt=False):
    q = problem["question"]
    correct = problem["correct_answer"]

    if domain == "gsm8k":
        if corrupt:
            # Generate a plausible but wrong answer
            try:
                wrong = str(int(correct) + random.choice([-1,1]) * random.randint(100, 2000))
            except:
                wrong = "999"
            system = f"""You are solving a math problem. Generate a step-by-step solution in EXACTLY {target_steps} steps.
IMPORTANT: The solution must arrive at the WRONG answer: {wrong}
Make the reasoning look plausible but introduce a subtle error.
After the steps, write "Final Answer: {wrong}"
Format: Step 1: ... Step 2: ... etc."""
        else:
            system = f"""You are solving a math problem. Generate a step-by-step solution in EXACTLY {target_steps} steps.
The correct answer is {correct}.
After the steps, write "Final Answer: {correct}"
Format: Step 1: ... Step 2: ... etc."""

    elif domain == "csqa":
        if corrupt:
            choices = list(problem.get("choices", {}).keys())
            wrong_choices = [c for c in choices if c != correct]
            wrong = random.choice(wrong_choices) if wrong_choices else "B"
            system = f"""You are solving a multiple-choice question. Generate reasoning in EXACTLY {target_steps} steps.
IMPORTANT: The reasoning must conclude with the WRONG answer: ({wrong})
Make the reasoning look plausible but subtly flawed.
After the steps, write "Final Answer: {wrong}"
Format: Step 1: ... Step 2: ... etc."""
        else:
            system = f"""You are solving a multiple-choice question. Generate reasoning in EXACTLY {target_steps} steps.
The correct answer is ({correct}).
After the steps, write "Final Answer: {correct}"
Format: Step 1: ... Step 2: ... etc."""

    else:  # boolq
        if corrupt:
            wrong = not correct
            system = f"""You are answering a yes/no question based on a passage. Generate reasoning in EXACTLY {target_steps} steps.
IMPORTANT: The reasoning must conclude with the WRONG answer: {wrong}
Make the reasoning look plausible but subtly flawed.
After the steps, write "Final Answer: {wrong}"
Format: Step 1: ... Step 2: ... etc."""
        else:
            system = f"""You are answering a yes/no question based on a passage. Generate reasoning in EXACTLY {target_steps} steps.
The correct answer is {correct}.
After the steps, write "Final Answer: {correct}"
Format: Step 1: ... Step 2: ... etc."""

    user = f"Solve this problem in exactly {target_steps} steps:\n\n{q}"
    return system, user

def parse_steps(text):
    steps = re.findall(r'Step \d+:.*?(?=Step \d+:|Final Answer:|$)', text, re.DOTALL)
    return [s.strip() for s in steps if s.strip()]

def extract_wrong_answer(problem, domain, corrupt_text):
    if domain == "gsm8k":
        match = re.search(r'Final Answer:\s*(-?[\d,]+)', corrupt_text)
        if match:
            return match.group(1).replace(",", "")
        nums = re.findall(r'-?[\d,]+', corrupt_text)
        return nums[-1].replace(",", "") if nums else "0"
    elif domain == "boolq":
        return not problem["correct_answer"]
    else:
        match = re.search(r'Final Answer:\s*([A-E])', corrupt_text)
        if match:
            ans = match.group(1)
            return ans if ans != problem["correct_answer"] else None
        return None

# ━━━ GENERATE TRACES ━━━
all_traces = {}
producer_provider, producer_model = PRODUCER

for domain in DOMAINS:
    all_traces[domain] = []
    problems = data[domain]
    print(f"\n{'='*50}")
    print(f"Generating traces for {domain} ({len(problems)} problems)...")

    for prob in tqdm(problems, desc=domain):
        # Clean trace
        sys_p, usr_p = make_trace_prompt(prob, domain, TRACE_LENGTH, corrupt=False)
        clean_text = call_api(producer_provider, producer_model, sys_p, usr_p, max_tokens=MAX_TOKENS_TRACE)
        clean_steps = parse_steps(clean_text)

        # Corrupt trace
        sys_p, usr_p = make_trace_prompt(prob, domain, TRACE_LENGTH, corrupt=True)
        corrupt_text = call_api(producer_provider, producer_model, sys_p, usr_p, max_tokens=MAX_TOKENS_TRACE)
        corrupt_steps = parse_steps(corrupt_text)

        wrong_answer = extract_wrong_answer(prob, domain, corrupt_text)
        if wrong_answer is None:
            wrong_answer = "UNKNOWN"

        trace_entry = {
            "domain": domain,
            "problem_index": prob["problem_index"],
            "correct_answer": prob["correct_answer"],
            "wrong_answer": wrong_answer,
            "clean_steps": clean_steps,
            "corrupt_steps": corrupt_steps,
            "n_clean": len(clean_steps),
            "n_corrupt": len(corrupt_steps),
        }
        all_traces[domain].append(trace_entry)

    print(f"  ✓ {domain}: {len(all_traces[domain])} traces generated")

# Save traces
with open(f"{BASE_DIR}/traces/all_traces.json", "w") as f:
    json.dump(all_traces, f, indent=2, default=str)
print(f"\n✓ All traces saved to {BASE_DIR}/traces/all_traces.json")


In [ ]:
# @title 5. Run Consumer Evaluation (Main Experiment)
# Runtime: ~2-3 hours for 3,750 trials

def mix_traces(clean_steps, corrupt_steps, lam, seed):
    """Mix clean and corrupt steps according to lambda.
    λ=0: all clean, λ=1: all corrupt, λ=0.4: 40% corrupt."""
    rng = random.Random(seed)
    if lam == 0 or not corrupt_steps:
        return clean_steps.copy(), 0
    if lam >= 1.0:
        return corrupt_steps.copy(), len(corrupt_steps)

    # Use the longer list as base length
    n = max(len(clean_steps), len(corrupt_steps))
    # Pad shorter list
    cs = clean_steps + [""] * (n - len(clean_steps))
    cr = corrupt_steps + [""] * (n - len(corrupt_steps))

    k = max(1, int(n * lam))
    indices = sorted(rng.sample(range(n), min(k, n)))

    result = cs.copy()
    for idx in indices:
        if idx < len(cr) and cr[idx]:
            result[idx] = cr[idx]

    result = [s for s in result if s]  # Remove empty
    n_corrupted = sum(1 for i in indices if i < len(cr) and cr[i])
    return result, n_corrupted

def build_prompt(problem, domain, trace_steps, wrong_answer):
    """Build the consumer prompt with reasoning trace and wrong cue."""
    trace_text = "\n".join([f"Step {i+1}: {s}" if not s.startswith("Step") else s
                            for i, s in enumerate(trace_steps)])
    trace_text += f"\n\nFinal Answer: {wrong_answer}"

    if domain == "gsm8k":
        system = "You are solving math problems. Review the reasoning trace and provide your final answer.\nOutput ONLY: {\"final\": <number>}"
        user = f"Problem:\n{problem['question']}\n\nReasoning trace:\n{trace_text}\n\nYour answer: {{\"final\": <number>}}"
    elif domain == "boolq":
        system = "You are answering yes/no questions. Review the reasoning trace and provide your final answer.\nOutput ONLY: {\"final\": \"True\" or \"False\"}"
        user = f"Problem:\n{problem['question']}\n\nReasoning trace:\n{trace_text}\n\nYour answer: {{\"final\": \"True\" or \"False\"}}"
    else:  # csqa
        system = "You are solving multiple-choice questions. Review the reasoning trace and provide your final answer.\nOutput ONLY: {\"final\": \"A\" or \"B\" etc.}"
        user = f"Problem:\n{problem['question']}\n\nReasoning trace:\n{trace_text}\n\nYour answer: {{\"final\": \"<letter>\"}}"

    return system, user

def build_direct_prompt(problem, domain):
    """Build a direct prompt (no trace) for baseline."""
    if domain == "gsm8k":
        system = "Solve this math problem. Output ONLY: {\"final\": <number>}"
        user = problem["question"]
    elif domain == "boolq":
        system = "Answer this yes/no question. Output ONLY: {\"final\": \"True\" or \"False\"}"
        user = problem["question"]
    else:
        system = "Answer this multiple-choice question. Output ONLY: {\"final\": \"<letter>\"}"
        user = problem["question"]
    return system, user

def answers_match(a, b, answer_type):
    if a is None or b is None:
        return False
    if answer_type == "numerical":
        try:
            return str(int(float(str(a).replace(",","")))) == str(int(float(str(b).replace(",",""))))
        except:
            return str(a).strip() == str(b).strip()
    elif answer_type == "boolean":
        return bool(a) == bool(b)
    else:
        return str(a).strip().upper() == str(b).strip().upper()

# ━━━ LOAD CHECKPOINT IF EXISTS ━━━
checkpoint_path = f"{BASE_DIR}/checkpoints/trials.json"
if os.path.exists(checkpoint_path):
    with open(checkpoint_path) as f:
        all_trials = json.load(f)
    completed = {(t["model"], t["domain"], t["problem_index"], t["condition"], str(t["lambda"]))
                 for t in all_trials}
    print(f"✓ Loaded checkpoint: {len(all_trials)} trials completed")
else:
    all_trials = []
    completed = set()
    print("Starting fresh (no checkpoint)")

# ━━━ RUN EXPERIMENT ━━━
total_expected = len(MODELS) * len(DOMAINS) * N_PROBLEMS * (1 + len(LAMBDA_LEVELS))
pbar = tqdm(total=total_expected - len(completed), desc="Trials")

for model_name, (provider, model_id) in MODELS.items():
    for domain in DOMAINS:
        problems = data[domain]
        traces = all_traces[domain]

        for prob_idx, (prob, trace) in enumerate(zip(problems, traces)):
            pidx = prob["problem_index"]
            answer_type = prob["answer_type"]

            # Direct baseline
            key = (model_name, domain, pidx, "direct", "None")
            if key not in completed:
                sys_p, usr_p = build_direct_prompt(prob, domain)
                raw = call_api(provider, model_id, sys_p, usr_p, max_tokens=MAX_TOKENS_CONSUMER)
                model_ans = parse_answer(raw, answer_type)
                is_correct = answers_match(model_ans, prob["correct_answer"], answer_type)

                all_trials.append({
                    "domain": domain, "problem_index": pidx,
                    "model": model_name, "condition": "direct",
                    "lambda": None, "correct_answer": prob["correct_answer"],
                    "model_answer": model_ans, "is_correct": is_correct,
                    "parse_method": "json", "raw_output": raw,
                    "timestamp": datetime.now().isoformat(),
                })
                completed.add(key)
                pbar.update(1)

            # Wrong-cue conditions at each lambda
            for lam in LAMBDA_LEVELS:
                key = (model_name, domain, pidx, "wrong_cue", str(lam))
                if key in completed:
                    continue

                mixed_steps, n_corrupt = mix_traces(
                    trace["clean_steps"], trace["corrupt_steps"],
                    lam, seed=GLOBAL_SEED + pidx * 100 + LAMBDA_LEVELS.index(lam)
                )
                sys_p, usr_p = build_prompt(prob, domain, mixed_steps, trace["wrong_answer"])
                raw = call_api(provider, model_id, sys_p, usr_p, max_tokens=MAX_TOKENS_CONSUMER)
                model_ans = parse_answer(raw, answer_type)
                is_correct = answers_match(model_ans, prob["correct_answer"], answer_type)
                followed_wrong = answers_match(model_ans, trace["wrong_answer"], answer_type)

                all_trials.append({
                    "domain": domain, "problem_index": pidx,
                    "model": model_name, "condition": "wrong_cue",
                    "lambda": lam, "correct_answer": prob["correct_answer"],
                    "wrong_answer": trace["wrong_answer"],
                    "model_answer": model_ans, "is_correct": is_correct,
                    "followed_wrong": followed_wrong,
                    "corruption_info": f"corrupted_{n_corrupt}_of_{len(mixed_steps)}",
                    "parse_method": "json", "raw_output": raw,
                    "timestamp": datetime.now().isoformat(),
                })
                completed.add(key)
                pbar.update(1)

            # Checkpoint every 50 trials
            if len(all_trials) % 50 == 0:
                with open(checkpoint_path, "w") as f:
                    json.dump(all_trials, f, default=str)

pbar.close()

# Final save
with open(checkpoint_path, "w") as f:
    json.dump(all_trials, f, indent=2, default=str)
print(f"\n✓ Complete: {len(all_trials)} trials saved")


In [ ]:

# ━━━ NumPy-safe JSON serialization ━━━
import numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.bool_,)):
            return bool(obj)
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


# @title 6. Compute Summary Statistics & CIF

with open(f"{BASE_DIR}/checkpoints/trials.json") as f:
    all_trials = json.load(f)

# Compute summary in E1b-compatible format
summary = {}

for model_name in MODELS:
    summary[model_name] = {}
    model_trials = [t for t in all_trials if t["model"] == model_name]

    for domain in DOMAINS:
        domain_trials = [t for t in model_trials if t["domain"] == domain]

        # Baseline accuracy
        direct = [t for t in domain_trials if t["condition"] == "direct"]
        baseline_acc = np.mean([t["is_correct"] for t in direct]) if direct else 0

        # Lambda sweep
        lambda_sweep = {}
        for lam in LAMBDA_LEVELS:
            lam_trials = [t for t in domain_trials
                          if t["condition"] == "wrong_cue" and t["lambda"] == lam]
            if not lam_trials:
                continue

            accuracy = np.mean([t["is_correct"] for t in lam_trials])

            # CIF: problems where model was correct at baseline but wrong with contamination
            cif_count = 0
            eligible = 0
            for lt in lam_trials:
                pidx = lt["problem_index"]
                baseline = [t for t in direct if t["problem_index"] == pidx]
                if baseline and baseline[0]["is_correct"]:
                    eligible += 1
                    if not lt["is_correct"]:
                        cif_count += 1

            cif_rate = cif_count / eligible if eligible > 0 else 0

            lambda_sweep[str(lam)] = {
                "n": len(lam_trials),
                "accuracy": round(accuracy, 4),
                "followed_wrong_rate": round(
                    np.mean([t.get("followed_wrong", False) for t in lam_trials]), 4
                ),
                "cif_rate": round(cif_rate, 4),
                "n_cif": cif_count,
                "n_eligible": eligible,
            }

        summary[model_name][domain] = {
            "baseline_accuracy": round(baseline_acc, 4),
            "lambda_sweep": lambda_sweep,
        }

# Save
with open(f"{BASE_DIR}/results/summary.json", "w") as f:
    json.dump(summary, f, indent=2, cls=NumpyEncoder)

# Print results table
print(f"{'Model':15s} {'Domain':8s} {'Baseline':>8s} | {'λ=0':>6s} {'λ=0.4':>6s} {'λ=0.8':>6s} {'λ=1.0':>6s}")
print("-" * 75)
for model in MODELS:
    for domain in DOMAINS:
        d = summary[model][domain]
        bl = f"{d['baseline_accuracy']:.0%}"
        cifs = []
        for lam in ["0.0", "0.4", "0.8", "1.0"]:
            if lam in d["lambda_sweep"]:
                cifs.append(f"{d['lambda_sweep'][lam]['cif_rate']:.0%}")
            else:
                cifs.append("  -")
        print(f"{model:15s} {domain:8s} {bl:>8s} | {' '.join(cifs)}")
    print()

print(f"\n✓ Summary saved to {BASE_DIR}/results/summary.json")


In [ ]:

# ━━━ NumPy-safe JSON serialization ━━━
import numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.bool_,)):
            return bool(obj)
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


# @title 7. Export for Merge with E1/E1b

# Create export in same format as A3_E1_export.json / A3_E1b_export.json
export = {
    "experiment_id": EXPERIMENT_ID,
    "date": datetime.now().strftime("%Y%m%d"),
    "problem_range": [PROBLEM_OFFSET, PROBLEM_OFFSET + N_PROBLEMS - 1],
    "seed": GLOBAL_SEED,
    "n_problems_per_domain": N_PROBLEMS,
    "models": list(MODELS.keys()),
    "domains": DOMAINS,
    "lambda_levels": LAMBDA_LEVELS,
    "summary": summary,
    "n_trials": len(all_trials),
}

with open(f"{BASE_DIR}/results/{EXPERIMENT_ID}_export.json", "w") as f:
    json.dump(export, f, indent=2, cls=NumpyEncoder)

# Also save config
config = {
    "experiment_id": EXPERIMENT_ID,
    "experiment_name": "E1 Augmentation (n=50→100)",
    "date": datetime.now().strftime("%Y%m%d"),
    "seed": GLOBAL_SEED,
    "n_problems_per_domain": N_PROBLEMS,
    "problem_offset": PROBLEM_OFFSET,
    "trace_length": TRACE_LENGTH,
    "models": [[name, *spec] for name, spec in MODELS.items()],
    "domains": DOMAINS,
    "producer_model": list(PRODUCER),
    "lambda_levels": LAMBDA_LEVELS,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS_CONSUMER,
    "checkpoint_interval": 50,
}
with open(f"{BASE_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2, cls=NumpyEncoder)

# Create combined summary (merge-ready)
print("━" * 60)
print("MERGE INSTRUCTIONS")
print("━" * 60)
print(f"""
To merge with E1/E1b data:

1. Download this folder: {BASE_DIR}
2. In analysis notebook, load both summaries:
   - E1b combined_summary.json (n=50, indices 0-49)
   - This export (n=50, indices 50-99)
3. Average CIF rates across both sets for n=100

Key files:
  {BASE_DIR}/results/{EXPERIMENT_ID}_export.json  ← merge data
  {BASE_DIR}/results/summary.json                ← standalone results
  {BASE_DIR}/checkpoints/trials.json             ← raw trial data
  {BASE_DIR}/config.json                         ← configuration
""")

# Quick comparison with E1/E1b ranges (if available)
print("\n✓ Export complete. Ready for merge.")


In [ ]:
# @title 8. Quick Visualization (Optional)
import matplotlib.pyplot as plt

MODELS_ORDER = ["sonnet4", "gpt4o", "haiku35", "gpt4omini", "gpt35"]
STYLES = {
    "sonnet4":   {"color": "#1565C0", "marker": "o", "label": "Sonnet 4"},
    "gpt4o":     {"color": "#FF8F00", "marker": "s", "label": "GPT-4o"},
    "haiku35":   {"color": "#7B1FA2", "marker": "D", "label": "Haiku 3.5"},
    "gpt4omini": {"color": "#2E7D32", "marker": "^", "label": "GPT-4o-mini"},
    "gpt35":     {"color": "#C62828", "marker": "v", "label": "GPT-3.5"},
}

fig, ax = plt.subplots(figsize=(10, 6))
x = np.array([0, 1, 2])
DOMAIN_LABELS = ["GSM8K\n(high verif.)", "CSQA\n(medium)", "BoolQ\n(low verif.)"]

for model in MODELS_ORDER:
    if model not in summary:
        continue
    peak_cifs = []
    for domain in DOMAINS:
        d = summary[model][domain]
        cifs = [v["cif_rate"] for v in d["lambda_sweep"].values()]
        peak_cifs.append(max(cifs) if cifs else 0)

    s = STYLES[model]
    ax.plot(x, peak_cifs, marker=s["marker"], color=s["color"],
            markersize=9, linewidth=2.2, label=s["label"],
            markeredgecolor="white", markeredgewidth=1.5)

ax.set_xticks(x)
ax.set_xticklabels(DOMAIN_LABELS)
ax.set_ylabel("Peak CIF (Backfire Rate)")
ax.set_title(f"E1-Aug Results (n={N_PROBLEMS}, indices {PROBLEM_OFFSET}-{PROBLEM_OFFSET+N_PROBLEMS-1})")
ax.legend()
ax.grid(True, alpha=0.2)
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)

plt.tight_layout()
plt.savefig(f"{BASE_DIR}/figures/E1aug_phase_diagram.png", dpi=150)
plt.show()
print("✓ Figure saved")
